# dataloader-pin-memory-workers — worked example 1: Configure drop_last and predict the batch count

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataloader-pin-memory-workers`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A `DataLoader` slices a dataset into batches of `batch_size`. With `drop_last=False` the final, possibly-partial batch is kept, so the loader yields `ceil(N / batch_size)` batches; with `drop_last=True` the partial tail is discarded, yielding `floor(N / batch_size)` batches. This matters for GPU throughput because partial batches under-utilize the device and can break shape-rigid model code.

## Worked solution

We want a loader that *drops* the ragged last batch so every batch is exactly full.

1. **Build a small dataset.** `TensorDataset(X)` wraps a tensor so item `i` is the tuple `(X[i],)`. We use `N=23` items of shape `(4,)` so the batching is non-trivial.
2. **Construct the loader with `drop_last=True`.** We also set `num_workers=0` (single-process, notebook-safe) and `pin_memory=False` (CPU test box). `shuffle=False` keeps the count deterministic.
3. **Why `floor`?** With `batch_size=5` and `N=23`, full batches are `23 // 5 = 4` (covering 20 items); the trailing 3 items can't fill a 5th batch, and `drop_last=True` throws them away. So we expect 4 batches.
4. **Verify by iterating.** We count batches and confirm each has exactly 5 rows. Counting empirically is the honest check — never trust the formula without iterating once.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def make_drop_last_loader(dataset, batch_size):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=0,
        pin_memory=False,
        shuffle=False,
        drop_last=True,
    )

t.manual_seed(0)
X = t.arange(23 * 4, dtype=t.float32).reshape(23, 4)
ds = TensorDataset(X)
loader = make_drop_last_loader(ds, batch_size=5)

batch_sizes = [b[0].shape[0] for b in loader]
import math
expected = math.floor(len(ds) / 5)
print('num batches:', len(batch_sizes), '(expected', expected, ')')
print('batch sizes:', batch_sizes)